# Verification Notebook V3: Viral Validation

**Claim**: See `paper/manifest.yaml`::R3

**Runtime**: ~2 minutes

This notebook verifies viral validation claims: correlation with phylogenetic depth (ρ = 0.84), null controls, substrate independence.

In [ ]:
import yaml
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
import json
from scipy.stats import pearsonr

# Load manifest
manifest_path = Path('../paper/manifest.yaml')
manifest = yaml.safe_load(manifest_path.open())
result = manifest['results']['R3']  # R3: Viral evolution validates curvature-entropy law

print(f"Verifying: {result['title']}")
print(f"Result ID: R3")
print(f"Category: viral-measurements")

## Load Canonical Data

In [ ]:
# Load data from canonical outputs
data_path = Path('../data/outputs/viral_measurements/')

if not data_path.exists():
    print(f"Data directory not found: {data_path}")
    print("   Run: python scripts/populate_outputs.py")
else:
    print(f"Data directory found: {data_path}")
    
# List available files
if data_path.exists():
    files = list(data_path.glob('*'))
    print(f"\nAvailable files:")
    for f in files:
        print(f"  - {f.name}")

## Verify Claims

In [ ]:
# Load viral measurements
viral_csv = data_path / 'kappa_per_virus.csv'
null_controls_json = data_path / 'null_control_results.json'

# Check 1: Correlation with phylogenetic depth
print("\nCheck 1: Correlation with phylogenetic depth")
if viral_csv.exists():
    viral_df = pd.read_csv(viral_csv)
    # Filter out controls
    viral_df_filtered = viral_df[viral_df['note'] != 'control'].copy()
    
    # Define phylogenetic depth (years) - approximate from literature
    depth_map = {
        'Zika': 10,
        'West_Nile': 30,
        'Yellow_Fever': 100,
        'SARS-CoV-2': 4,
        'Influenza_A': 100,
        'HCV': 200,
        'DENV': 500,
        'HIV-1_subtype': 40,
        'Poliovirus': 100,
        'Enterovirus': 200,
        'Measles': 1500,
        'Mumps': 500,
        'Rabies': 1000
    }
    
    viral_df_filtered['depth_years'] = viral_df_filtered['virus'].map(depth_map)
    viral_df_filtered = viral_df_filtered.dropna(subset=['depth_years'])
    
    if len(viral_df_filtered) >= 3:
        kappa_vals = viral_df_filtered['kappa'].values
        depth_vals = np.log10(viral_df_filtered['depth_years'].values)  # Log scale
        r, p = pearsonr(kappa_vals, depth_vals)
        
        expected_r = 0.80
        expected_p = 0.01
        passed_1 = r > expected_r and p < expected_p
        
        print(f"  Pearson r: {r:.4f}")
        print(f"  p-value: {p:.4f}")
        print(f"  Expected: r > {expected_r:.2f}, p < {expected_p:.2f}")
        print(f"  Status: {'PASS' if passed_1 else 'FAIL'}")
    else:
        passed_1 = False
        print(f"  Status: FAIL (insufficient data)")
else:
    passed_1 = False
    print(f"  Status: FAIL (file not found)")

# Check 2: Null controls - label shuffling effect
print("\nCheck 2: Null controls - label shuffling")
if null_controls_json.exists():
    null_controls = json.load(open(null_controls_json))
    authentic_range = null_controls['authentic_labels']['hex_range']
    shuffled_range = null_controls['label_shuffling']['hex_range']
    reduction = null_controls['label_shuffling']['hex_range_reduction']
    
    # Check that reduction is substantial (> 50%)
    expected_reduction = 0.50
    passed_2 = reduction > expected_reduction
    
    print(f"  Authentic HEX range: {authentic_range:.2f}")
    print(f"  Shuffled HEX range: {shuffled_range:.2f}")
    print(f"  Reduction: {reduction:.2f} ({reduction*100:.0f}%)")
    print(f"  Expected: > {expected_reduction*100:.0f}%")
    print(f"  Status: {'PASS' if passed_2 else 'FAIL'}")
else:
    passed_2 = False
    print(f"  Status: FAIL (file not found)")

# Check 3: Substrate independence (DNA vs RNA)
print("\nCheck 3: Substrate independence")
if viral_csv.exists():
    viral_df = pd.read_csv(viral_csv)
    dna_viruses = viral_df[viral_df['type'] == 'DNA']
    rna_viruses = viral_df[viral_df['type'] == 'RNA']
    
    if len(dna_viruses) > 0 and len(rna_viruses) > 0:
        dna_kappa = dna_viruses['kappa'].values
        rna_kappa = rna_viruses['kappa'].values
        
        # Check overlap: DNA viruses should overlap RNA range
        dna_min, dna_max = dna_kappa.min(), dna_kappa.max()
        rna_min, rna_max = rna_kappa.min(), rna_kappa.max()
        
        # Overlap exists if DNA range intersects RNA range
        overlap = not (dna_max < rna_min or dna_min > rna_max)
        passed_3 = overlap
        
        print(f"  DNA viruses κ range: [{dna_min:.2f}, {dna_max:.2f}]")
        print(f"  RNA viruses κ range: [{rna_min:.2f}, {rna_max:.2f}]")
        print(f"  Overlap: {overlap}")
        print(f"  Status: {'PASS' if passed_3 else 'FAIL'}")
    else:
        passed_3 = False
        print(f"  Status: FAIL (insufficient data)")
else:
    passed_3 = False
    print(f"  Status: FAIL (file not found)")

# Compile results
# Initialize variables for results compilation
r_val = None
p_val = None
reduction_val = None
overlap_val = None

if viral_csv.exists() and 'viral_df_filtered' in locals() and len(viral_df_filtered) >= 3:
    r_val = float(r)
    p_val = float(p)

if null_controls_json.exists() and 'reduction' in locals():
    reduction_val = reduction

if viral_csv.exists() and 'dna_viruses' in locals() and len(dna_viruses) > 0 and 'overlap' in locals():
    overlap_val = overlap

verified_checks = [
    {'name': 'correlation_phylogenetic_depth', 'expected': 'ρ > 0.80, p < 0.01', 'passed': passed_1, 'value': {'r': r_val, 'p': p_val} if r_val is not None else None},
    {'name': 'null_controls_label_shuffling', 'expected': '> 50% reduction', 'passed': passed_2, 'value': reduction_val},
    {'name': 'substrate_independence', 'expected': 'DNA and RNA overlap', 'passed': passed_3, 'value': overlap_val}
]

all_passed = all(c['passed'] for c in verified_checks)
print(f"\n{'='*60}")
print(f"Overall: {'PASS' if all_passed else 'FAIL'}")
print(f"{'='*60}")

## Update Results

In [ ]:
# Update results.yaml with verification status
results_path = Path('../paper/results.yaml')

if results_path.exists():
    data = yaml.safe_load(results_path.open())
    # Handle both old structure (top-level) and new structure (results key)
    if 'results' in data:
        results = data['results']
    else:
        results = {k: v for k, v in data.items() if k.startswith('R')}
else:
    results = {}

if 'R3' not in results:
    results['R3'] = {}

results['R3']['verified'] = all_passed
results['R3']['verification_date'] = datetime.now().isoformat()
results['R3']['checks'] = verified_checks

# Save updated results with proper structure
output = {'results': results}
with results_path.open('w') as f:
    yaml.dump(output, f, default_flow_style=False, sort_keys=False, allow_unicode=True)

print(f"\nResults updated in {results_path}")
print(f"  Verified: {all_passed}")
print(f"  Date: {results['R3']['verification_date']}")